# Урок 5.3. Ollama в продакшене: как обернуть демон в надёжный сервис

Модуль 5 «Локальные LLM: запуск без облака и без GPU» · урок 3 из 7

Ollama работает у нас с урока 1.5 — и уже успела преподать пару уроков надёжности.
В уроке 3.2 она **самообновилась посреди работы**: инсталлер убил API, и только
`/healthz` нашего сервиса честно отыграл `degraded → ok`. А перезапуск Docker Desktop
однажды унёс контейнер Qdrant вместе с коллекциями. Десктопное приложение, которое
живёт в трее пользователя, — это не сервис. Сегодня чиним.

## 🎯 Что вы узнаете

- Как устроена Ollama внутри: демон, REST API, реестр моделей — «Docker для LLM»
- Три способа запуска как службы: NSSM на Windows, systemd на Linux, Docker
- Что делает `keep_alive` и почему без него каждый запрос платит за холодный старт
- Как работают параллелизм и очередь (`OLLAMA_NUM_PARALLEL`, `OLLAMA_MAX_QUEUE`) и почему на CPU параллелизм почти не помогает
- Как писать надёжный клиент: таймауты, матрица retry, экспоненциальный backoff с джиттером


## Архитектура Ollama: это Docker для моделей

Ollama — не библиотека, а **клиент-серверная система** поверх движка llama.cpp:

```
ваш код (httpx) ──┐
                  ├──HTTP──> демон ollama serve (REST API, 127.0.0.1:11434)
ollama run/ls ────┘              │                        │
(CLI — тоже HTTP-клиент)         ▼                        ▼
                         runner-процесс           хранилище моделей
                         (llama.cpp, модель       (blobs + manifests,
                          в RAM/VRAM)              ~/.ollama/models)
                                                          ▲
                                    ollama pull ──────────┘ (реестр ollama.com)
```

Параллели с известными вам системами:

| Ollama | Docker | PostgreSQL |
|---|---|---|
| демон `ollama serve` | dockerd | postgres-сервер |
| `ollama pull qwen2.5:3b` | `docker pull nginx:1.27` | — |
| модель = manifest + blobs (слои) | образ = manifest + layers | — |
| REST API на :11434 | Docker API | протокол на :5432 |
| runner с моделью в RAM | контейнер | worker-процесс с shared buffers |
| `keep_alive` | — | пул соединений: держим тёплым |

CLI-команды (`ollama run`, `ollama ls`) — это просто HTTP-клиенты к демону. Всё,
что может CLI, ваш Python-код может через REST API.

**Главное продовое следствие**: у Ollama **нет аутентификации**. Кто может
постучаться на порт 11434 — тот может пользоваться моделью и выгружать/загружать
модели. Поэтому демон слушает только `127.0.0.1`, а наружу его выставляют
исключительно через reverse-proxy с auth (или наш прокси из урока 5.7).


## Установка и первый контакт

**Windows**: установщик с <https://ollama.com/download> или `winget install Ollama.Ollama`. После установки в трее появится иконка — демон уже слушает `127.0.0.1:11434`.

**Linux**: `curl -fsSL https://ollama.com/install.sh | sh` — скрипт сразу создаёт systemd-службу.

Проверка (в любом терминале):

```powershell
# PowerShell
ollama --version
Invoke-RestMethod http://127.0.0.1:11434/api/version
```

```bash
# bash
ollama --version
curl -s http://127.0.0.1:11434/api/version
```

Где живут модели по умолчанию: Windows — `C:\Users\<вы>\.ollama\models`, Linux — `/usr/share/ollama/.ollama/models` (для службы). Переносится переменной `OLLAMA_MODELS` — пригодится, когда системный диск маленький.

Теперь тот же «первый контакт» из Python:

In [ ]:
import httpx

# Именно 127.0.0.1, не localhost — почему, замерим в следующей ячейке
OLLAMA_URL = "http://127.0.0.1:11434"
NS = 1_000_000_000


def ollama_available() -> bool:
    try:
        return httpx.get(f"{OLLAMA_URL}/api/version", timeout=2).status_code == 200
    except httpx.HTTPError:
        return False


OLLAMA_UP = ollama_available()
if OLLAMA_UP:
    print("Версия демона:", httpx.get(f"{OLLAMA_URL}/api/version", timeout=5).json())
else:
    print("Ollama не запущена — сетевые ячейки будут пропущены")

### Мини-замер: `localhost` против `127.0.0.1`

Старая грабля из урока 3.2, теперь с цифрами. На Windows `localhost` резолвится
сначала в IPv6 `::1`, которого Ollama не слушает, — и каждое **свежее** соединение
платит за таймаут IPv6-попытки, прежде чем откатиться на IPv4:


In [ ]:
import time

if OLLAMA_UP:
    for host in ("localhost", "127.0.0.1"):
        times = []
        for _ in range(3):
            t0 = time.perf_counter()
            with httpx.Client() as probe:  # свежий клиент = свежее TCP-соединение
                probe.get(f"http://{host}:11434/api/version", timeout=10)
            times.append(time.perf_counter() - t0)
        print(f"{host:<12} свежие соединения: " + ", ".join(f"{t * 1000:.0f} мс" for t in times))
else:
    print("Пропущено: Ollama не запущена")

У меня: `localhost` — 2.2–3 секунды на соединение, `127.0.0.1` — около секунды
(и та в основном на создание нового `httpx.Client`; на Linux разницы почти нет).
Боевой клиент ниже держит **один** `httpx.Client` на всё время жизни — соединение
переиспользуется, и штраф платится один раз. Но адрес всё равно пишем цифрами:
бесплатная страховка.

Теперь посмотрим, что демон знает о моделях — на диске и в памяти:


In [ ]:
if OLLAMA_UP:
    # Что скачано на диск (аналог docker images)
    tags = httpx.get(f"{OLLAMA_URL}/api/tags", timeout=5).json()
    for m in tags["models"]:
        print(f"на диске: {m['name']:<24} {m['size'] / 1024**3:.1f} ГБ  digest={m['digest'][:12]}")

    # Что сейчас загружено в RAM (аналог docker ps)
    ps = httpx.get(f"{OLLAMA_URL}/api/ps", timeout=5).json()
    if ps["models"]:
        for m in ps["models"]:
            print(f"в памяти: {m['name']:<24} выгрузится в {m.get('expires_at', '?')}")
    else:
        print("в памяти: пусто (первый запрос заплатит за холодный старт)")
else:
    print("Пропущено: Ollama не запущена")

## Демон как служба: три способа

Десктопное приложение Ollama живёт в сессии пользователя: разлогинились или упало — сервис умер. Для продакшена нужен супервизор, который стартует демон при загрузке ОС и перезапускает при падении, — та же логика, что `restart: unless-stopped` у контейнеров.

### Способ 1: Windows-служба через NSSM

Windows умеет запускать как службу только специально написанные программы; **NSSM** (Non-Sucking Service Manager) — обёртка, которая превращает в службу любой exe. План:

1. Установить NSSM: `winget install NSSM.NSSM`.
2. **Выключить автозапуск десктопного Ollama** (иконка в трее → Quit; Параметры → Приложения → Автозагрузка), иначе служба не займёт порт.
3. Из PowerShell **от администратора**:

```powershell
nssm install OllamaService "$env:LOCALAPPDATA\Programs\Ollama\ollama.exe" serve

# Переменные окружения демона. ВАЖНО: служба работает под LocalSystem,
# у которого свой профиль (C:\Windows\system32\config\systemprofile) —
# без явного OLLAMA_MODELS модели «переедут» туда.
nssm set OllamaService AppEnvironmentExtra `
    OLLAMA_HOST=127.0.0.1:11434 `
    OLLAMA_MODELS=C:\ollama\models `
    OLLAMA_KEEP_ALIVE=30m `
    OLLAMA_NUM_PARALLEL=1 `
    OLLAMA_MAX_QUEUE=128

# Логи с ротацией + автоперезапуск при падении
nssm set OllamaService AppStdout C:\ollama\logs\ollama.log
nssm set OllamaService AppStderr C:\ollama\logs\ollama-err.log
nssm set OllamaService AppRotateFiles 1
nssm set OllamaService AppRotateBytes 10485760
nssm set OllamaService Start SERVICE_AUTO_START
nssm set OllamaService AppExit Default Restart
nssm set OllamaService AppRestartDelay 5000

nssm start OllamaService
Get-Service OllamaService   # должно быть Running
```

Готовый скрипт со всеми этими командами и проверками — `deploy/install-ollama-service.ps1`. Удаление: `nssm stop OllamaService; nssm remove OllamaService confirm`.

### Способ 2: systemd на Linux

Официальный установщик уже создал юнит `ollama.service`. Добавить переменные окружения правильно так (не редактируя оригинальный юнит):

```bash
sudo systemctl edit ollama        # создаст override.conf
```

```ini
[Service]
Environment="OLLAMA_KEEP_ALIVE=30m"
Environment="OLLAMA_NUM_PARALLEL=1"
Environment="OLLAMA_MAX_QUEUE=128"
```

```bash
sudo systemctl daemon-reload && sudo systemctl restart ollama
systemctl status ollama && journalctl -u ollama -n 50   # статус и логи
```

Полный пример юнита «с нуля» (включая `MemoryMax` — потолок памяти через cgroups) — в `deploy/ollama.service`.

### Способ 3: Docker

`docker-compose.yml` лежит в корне модуля: том для моделей, healthcheck, `restart: unless-stopped`, лимит памяти. Запуск:

```powershell
docker compose up -d
docker compose exec ollama ollama pull qwen2.5:3b
```

Нюанс Windows: Docker Desktop работает поверх WSL2, у которой свой лимит RAM (обычно 50% хоста). Настройте `%USERPROFILE%\.wslconfig` → `[wsl2]`, `memory=12GB`, иначе контейнер с 7B-моделью упрётся в потолок WSL, а не хоста.

**Что выбрать?** Один физический Windows-хост → NSSM. Linux-сервер → systemd (или Docker, если вся инфраструктура уже в компоузах). Kubernetes/оркестрация → образ `ollama/ollama`.

## Переменные окружения, которые надо знать наизусть

Дефолты ниже — не из документации, а **из лога моего демона** (Ollama 0.32.x);
между версиями они меняются, поэтому первым делом научимся их проверять.

| Переменная | Дефолт (0.32.x) | Что делает |
|---|---|---|
| `OLLAMA_HOST` | `127.0.0.1:11434` | Адрес API. `0.0.0.0` — доступ из сети (только за reverse-proxy с auth!) |
| `OLLAMA_MODELS` | `~/.ollama/models` | Каталог моделей. Выносите на быстрый диск с местом |
| `OLLAMA_KEEP_ALIVE` | `5m` | Сколько держать модель в RAM после последнего запроса |
| `OLLAMA_NUM_PARALLEL` | `1` (в старых версиях — авто 4/1) | Сколько запросов один runner обрабатывает одновременно |
| `OLLAMA_MAX_LOADED_MODELS` | `0` = авто | Сколько разных моделей могут жить в RAM одновременно |
| `OLLAMA_MAX_QUEUE` | `512` | Длина очереди ожидания; при переполнении клиент получает **503** |
| `OLLAMA_CONTEXT_LENGTH` | `0` = авто (4096) | Контекст по умолчанию, если запрос не задал `num_ctx` |
| `OLLAMA_LOAD_TIMEOUT` | `5m` | Максимум на загрузку модели с диска |
| `OLLAMA_DEBUG` | `INFO` | Уровень логов (для разбора инцидентов — `DEBUG`) |

**Как проверить, с чем реально запущен демон**: первая строка лога старта содержит
`msg="server config"` и полный дамп настроек. Windows —
`%LOCALAPPDATA%\Ollama\server.log`, Linux — `journalctl -u ollama`. Фрагмент
из моего лога (сокращён):

```
time=2026-08-22T12:30:39 level=INFO source=routes.go msg="server config"
env="map[... OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_KEEP_ALIVE:5m0s
OLLAMA_MAX_QUEUE:512 OLLAMA_NUM_PARALLEL:1
OLLAMA_MODELS:C:\\Users\\rusla\\.ollama\\models ...]"
```

Эта привычка — «не верю докам, смотрю конфиг работающего процесса» — та же,
что `SHOW ALL;` в PostgreSQL.

## keep_alive: кеш с TTL, только для модели

Загрузка модели с диска в память — секунды для 2-ГБ файла на NVMe и десятки секунд
для 5-ГБ на обычном SSD (а с HDD — минуты). Если модель выгружается после каждого
запроса, **каждый** пользователь платит этот «холодный старт» — как если бы пул
соединений закрывал коннект после каждого запроса.

`keep_alive` управляет временем жизни модели в RAM после запроса — это TTL кеша:

- `"5m"`, `"2h"` — держать N после последнего запроса (каждый запрос сбрасывает таймер);
- `0` — выгрузить сразу (полезно для разовых батчей);
- `-1` — держать вечно.

Задаётся глобально (`OLLAMA_KEEP_ALIVE`) или на каждый запрос (поле `keep_alive`
в JSON — приоритетнее). Стратегии:

| Паттерн трафика | keep_alive |
|---|---|
| Постоянный поток запросов | `30m`–`1h` или `-1` |
| Редкие запросы, RAM нужна другим | дефолтные `5m` |
| Ночной батч раз в сутки | `0` — отработал и освободил память |
| Один выделенный LLM-сервер | `-1` + прогрев после рестарта |

Измерим холодный и тёплый старт (поле `load_duration` в ответе). Чтобы холодный
старт был честным, сначала принудительно выгрузим модель — запрос с `keep_alive: 0`
и без промпта означает «выгрузи сейчас»:


In [ ]:
MODEL = "qwen3.5:9b"  # рабочая модель курса


def generate_and_report(prompt: str, keep_alive: str = "5m") -> None:
    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "keep_alive": keep_alive,
        "options": {"num_predict": 30},
    }
    resp = httpx.post(f"{OLLAMA_URL}/api/generate", json=payload, timeout=300)
    resp.raise_for_status()
    d = resp.json()
    tps = d["eval_count"] / (d["eval_duration"] / NS) if d.get("eval_duration") else 0
    print(f"загрузка модели: {d.get('load_duration', 0) / NS:6.2f} с | "
          f"всего: {d.get('total_duration', 0) / NS:6.2f} с | "
          f"генерация: {tps:5.1f} ток/с")


if OLLAMA_UP:
    # принудительная выгрузка: без неё «холодный» старт может оказаться тёплым
    httpx.post(f"{OLLAMA_URL}/api/generate", json={"model": MODEL, "keep_alive": 0}, timeout=60)
    print("1-й запрос (холодный старт):")
    generate_and_report("Скажи «привет» одним словом.")
    print("2-й запрос (модель уже в RAM):")
    generate_and_report("Скажи «пока» одним словом.")
else:
    print("Пропущено: Ollama не запущена")

У меня холодный старт — **1.8–3.7 секунды** загрузки в разных прогонах (2-ГБ файл,
NVMe, модель едет в VRAM; разброс — файловый кеш ОС) против **~0.01 с** у тёплого. Масштабируйте на свои условия: 8B-файл
в 4.9 ГБ на SATA-SSD — уже 15–30 секунд, и **каждый** такой старт — это чей-то
запрос, висящий в ожидании. В уроке 4.3 мы видели это в проде: первый запрос
после простоя занимал ~37 секунд против ~2 у тёплых — потому и попал в мониторинг
как отдельная метрика.

Правило: keep_alive подбирается под паттерн трафика (таблица выше), а после
рестарта демона сервис **прогревают** — упражнение 1.


## Параллелизм и очередь: worker pool с bounded queue

Внутри Ollama устроена как знакомый вам паттерн «пул воркеров + ограниченная очередь» (uvicorn workers, пул соединений БД):

```
запросы --> [очередь <= OLLAMA_MAX_QUEUE] --> [слоты параллелизма: OLLAMA_NUM_PARALLEL] --> модель
                     |                                    
                переполнена --> HTTP 503 «server busy»
```

Три правила, которые уберегут от сюрпризов:

1. **Каждый слот параллелизма получает свой контекст**, то есть свой KV-кеш: `NUM_PARALLEL=4` при контексте 8k означает память под 4 × 8k токенов. Память умножается!
2. **На CPU параллелизм почти не увеличивает пропускную способность.** Генерация упирается в пропускную способность RAM (урок 2), и она одна на всех: два параллельных запроса поедут каждый вдвое медленнее. Параллелизм на CPU — про честное разделение latency, а не про throughput. Рабочая рекомендация: `NUM_PARALLEL=1` (максимум 2).
3. **Очередь — это защита, а не бесплатный буфер.** Запрос, простоявший в очереди 5 минут, скорее всего уже никому не нужен (клиент отвалился по таймауту). Лучше короткая очередь + быстрый 503 + retry на клиенте, чем бесконечное ожидание.

Ещё один тонкий момент: параметры, влияющие на память (`num_ctx`, `num_gpu`, ...),
заставляют Ollama **перезапустить runner** с новыми настройками — вы внезапно
платите холодный старт посреди дня. Хуже того, новый конфиг **остаётся жить**:
в уроке 5.2 после замера с `num_gpu: 0` модель продолжала считать на CPU даже
для запросов без опций, пока мы её не выгрузили. Опции живут с runner'ом,
а не с запросом. В проде зафиксируйте один набор опций для всех клиентов.

## Надёжный клиент: таймауты и матрица retry

Ollama — обычная HTTP-зависимость, и к ней применимы все правила работы с нестабильным апстримом. Особенность одна: **генерация легально может идти минуты**, поэтому таймауты должны различать «не могу подключиться» (быстро) и «долго генерирует» (нормально).

Матрица решений — что делать с каждой ошибкой:

| Ситуация | Retry? | Почему |
|---|---|---|
| `ConnectError` / `ConnectTimeout` | ✅ с backoff | Демон перезапускается — станет живым через секунды |
| HTTP 503 (очередь полна) | ✅ с backoff + джиттер | Перегрузка временная; джиттер размазывает толпу |
| HTTP 5xx прочие | ✅ 1–2 раза | Может быть временный сбой runner |
| HTTP 404 (нет модели) | ❌ | Повторение не скачает модель; чините конфиг |
| HTTP 400 (кривой запрос) | ❌ | Ошибка у вас в коде |
| `ReadTimeout` при генерации | ❌ (обычно) | Повторный запрос удвоит нагрузку — «retry storm»; лучше вернуть 504 и разбираться |

Backoff обязательно с **джиттером** (случайной добавкой): если 50 клиентов получили 503 одновременно и повторят через ровно 1 секунду — они снова прибегут толпой. Реализуем клиент на `httpx` + `tenacity`:

In [ ]:
from dataclasses import dataclass

import httpx
from tenacity import retry, retry_if_exception, stop_after_attempt, wait_exponential_jitter


class OllamaHTTPError(Exception):
    def __init__(self, status_code: int, detail: str):
        self.status_code = status_code
        super().__init__(f"HTTP {status_code}: {detail}")


def _is_retryable(exc: BaseException) -> bool:
    """Матрица retry: сетевые ошибки подключения и 502/503 — да, остальное — нет."""
    if isinstance(exc, (httpx.ConnectError, httpx.ConnectTimeout)):
        return True
    return isinstance(exc, OllamaHTTPError) and exc.status_code in (502, 503)


@dataclass
class GenerateResult:
    text: str
    prompt_tokens: int
    completion_tokens: int
    load_seconds: float
    tokens_per_second: float | None


class RobustOllamaClient:
    """Синхронный клиент Ollama с таймаутами и retry по матрице выше."""

    def __init__(self, base_url: str = OLLAMA_URL, read_timeout_s: float = 180.0):
        # Ключевой приём: connect-таймаут короткий, read — длинный.
        # «Не подключился за 5 с» — это авария; «генерирует 2 минуты» — это норма.
        self._client = httpx.Client(
            base_url=base_url,
            timeout=httpx.Timeout(connect=5.0, read=read_timeout_s, write=10.0, pool=5.0),
        )

    @retry(
        retry=retry_if_exception(_is_retryable),
        stop=stop_after_attempt(3),
        wait=wait_exponential_jitter(initial=0.5, max=8.0),  # 0.5s, ~1s, ~2s + джиттер
        reraise=True,
    )
    def _post(self, path: str, payload: dict) -> dict:
        resp = self._client.post(path, json=payload)
        if resp.status_code >= 400:
            raise OllamaHTTPError(resp.status_code, resp.text[:300])
        return resp.json()

    def generate(self, model: str, prompt: str, *, system: str | None = None,
                 keep_alive: str = "30m", **options) -> GenerateResult:
        payload = {"model": model, "prompt": prompt, "stream": False,
                   "keep_alive": keep_alive, "options": options}
        if system:
            payload["system"] = system
        d = self._post("/api/generate", payload)
        eval_count, eval_duration = d.get("eval_count", 0), d.get("eval_duration", 0)
        return GenerateResult(
            text=d["response"],
            prompt_tokens=d.get("prompt_eval_count", 0),
            completion_tokens=eval_count,
            load_seconds=d.get("load_duration", 0) / NS,
            tokens_per_second=eval_count / (eval_duration / NS) if eval_duration else None,
        )

    def close(self) -> None:
        self._client.close()

In [ ]:
if OLLAMA_UP:
    client = RobustOllamaClient()
    try:
        result = client.generate(
            MODEL,
            "Объясни одним предложением, что такое очередь сообщений.",
            system="Отвечай кратко и по-русски.",
            temperature=0.2,
            num_predict=60,
        )
        print(result.text)
        print(f"\n[{result.completion_tokens} токенов, "
              f"{result.tokens_per_second:.1f} ток/с, загрузка {result.load_seconds:.2f} с]")

        # Проверим матрицу retry: несуществующая модель -> 404, повторов быть НЕ должно
        t0 = time.perf_counter()
        try:
            client.generate("no-such-model:1b", "ping", num_predict=1)
        except OllamaHTTPError as exc:
            print(f"\n404-тест: {exc}")
            print(f"          упали за {time.perf_counter() - t0:.2f} с — мгновенно, без повторов")
    finally:
        client.close()
else:
    print("Пропущено: Ollama не запущена")

Этот клиент — заготовка боевого кода: в уроке 5.7 мы перенесём ту же логику в асинхронный FastAPI-прокси. Обратите внимание, чего в нём **нет**: retry на `ReadTimeout`. Если генерация не уложилась в 3 минуты — повтор сделает только хуже; это сигнал для алерта (урок 5.5) и разбора (урок 5.6).

## ⚠️ Частые ошибки

1. **`OLLAMA_HOST=0.0.0.0` без защиты.** У Ollama нет ни аутентификации, ни rate limit: открытый порт — это бесплатный LLM для всего офиса (или интернета) и OOM у вас. Только localhost или reverse-proxy с auth.
2. **Поднять `NUM_PARALLEL`, чтобы «стало быстрее».** На CPU станет медленнее каждому: пропускная способность RAM одна на всех, а KV-кеш умножится на число слотов — привет, OOM. Параллелизм — про латентность на GPU, не про CPU-throughput.
3. **Служба под LocalSystem качает модели «в никуда».** Без явного `OLLAMA_MODELS` служба Windows складывает гигабайты в профиль system32. Диск C: заканчивается, все удивляются.
4. **`keep_alive=-1` и несколько моделей.** «Вечные» модели не выгружаются сами; вторая-третья модель к ним не влезет, и запросы начнут ждать выгрузку/загрузку. Планируйте RAM: `MAX_LOADED_MODELS=1`, если модель одна и главная.
5. **Один общий таймаут на всё.** `timeout=10` убивает легитимную минутную генерацию, `timeout=300` заставляет ждать 5 минут при мёртвом демоне. Разделяйте connect (секунды) и read (минуты).
6. **Клиент ходит на `localhost` вместо `127.0.0.1`.** На Windows это лишние ~1–2 секунды на каждое свежее соединение (IPv6-попытка `::1`, которую Ollama не слушает) — мы измерили это выше. Та же грабля уже ловилась в уроке 3.2.


## Упражнения

1. **Прогрев после рестарта.** Напишите функцию `warmup(models: list[str])`, которая после старта демона загружает модели в RAM (запрос с `num_predict=1`) и печатает время загрузки каждой. Такой скрипт запускают после деплоя, чтобы первый пользователь не платил холодный старт.
2. **Ретрай с джиттером руками.** Без tenacity напишите декоратор `retry_with_jitter(attempts=3, base_delay=0.5)`, повторяющий функцию при `httpx.ConnectError` и `OllamaHTTPError(503)` с задержкой `base_delay * 2**attempt * random(0.5..1.5)`. Сравните с поведением tenacity.
3. **Конфиг под сценарий.** Сервер: 16 ГБ RAM, одна модель 7B-Q4 (~4.7 ГБ файл), постоянный поток запросов от внутреннего сервиса, клиенты умеют retry. Выпишите значения `OLLAMA_KEEP_ALIVE`, `OLLAMA_NUM_PARALLEL`, `OLLAMA_MAX_LOADED_MODELS`, `OLLAMA_MAX_QUEUE`, `OLLAMA_CONTEXT_LENGTH` и обоснуйте каждое.

Решения — ниже.

## Мини-квиз

<details><summary><b>1. Клиент получил от Ollama HTTP 503. Наиболее вероятная причина и правильная реакция?</b></summary>

Очередь запросов переполнена (`OLLAMA_MAX_QUEUE`) — демон жив, но перегружен.
Правильная реакция: подождать с экспоненциальным backoff + джиттером и повторить;
при повторных 503 — алерт о перегрузке (возможно, пора вторую ноду или меньшую модель).
</details>

<details><summary><b>2. Почему OLLAMA_NUM_PARALLEL=4 увеличивает потребление памяти? Что именно умножается?</b></summary>

Умножается **KV-кеш**: каждый слот параллелизма резервирует свой контекст. Память
под веса одна на всех, а вот контекстная память — по числу слотов; Ollama планирует
её при загрузке модели, даже если параллельных запросов сейчас нет.
</details>

<details><summary><b>3. Чем плох «ровный» retry через фиксированную 1 секунду у 50 клиентов одновременно?</b></summary>

Все 50 клиентов повторят запрос **в одну и ту же миллисекунду** и снова уронят
сервис — это «thundering herd». Джиттер (случайная добавка к задержке) размазывает
повторы во времени.
</details>

<details><summary><b>4. Служба Windows с Ollama работает, но ollama ls из вашего терминала показывает пустой список, хотя служба качала модели. Что произошло?</b></summary>

CLI смотрит на модели пользователя (`%USERPROFILE%\.ollama`), а служба под
LocalSystem — на свой профиль (или на `OLLAMA_MODELS`). Это разные каталоги.
Решение: один явный `OLLAMA_MODELS` для всех — и у службы, и в вашей сессии.
</details>

<details><summary><b>5. Какие два таймаута надо настраивать по-разному в клиенте LLM и почему?</b></summary>

**Connect-таймаут** короткий (секунды): недоступность демона надо обнаруживать
быстро. **Read-таймаут** длинный (минуты): генерация легально долгая. Один общий
таймаут не может быть одновременно и коротким, и длинным — поэтому
`httpx.Timeout(connect=5, read=180, ...)`.
</details>

<details><summary><b>6. После эксперимента коллега дёрнул модель с num_gpu=0, и теперь все запросы медленные. Что случилось и как починить?</b></summary>

Опции, влияющие на память, перезапускают runner, и новый конфиг **остаётся с ним**:
модель теперь считает на CPU для всех, включая запросы без опций. Починка — выгрузить
модель (`keep_alive: 0`) или дёрнуть запрос с явными «правильными» опциями. Профилактика —
один согласованный набор опций для всех клиентов.
</details>


## Итоги

- Ollama = демон с REST API поверх llama.cpp; модели хранятся как manifest+blobs — «Docker для LLM». Аутентификации нет — наружу только через прокси.
- Реальный конфиг демона — в строке `msg="server config"` лога старта; дефолты между версиями меняются.
- Служба: NSSM (Windows), systemd + drop-in (Linux), Docker с healthcheck и лимитом памяти. Всегда: автостарт, автоперезапуск, логи, явный `OLLAMA_MODELS`.
- `keep_alive` — TTL модели в RAM: подбирается под паттерн трафика, спасает от холодных стартов.
- Параллелизм на CPU не даёт throughput и умножает KV-кеш; очередь + 503 — это защита, настройте её осознанно.
- Клиент: короткий connect-таймаут, длинный read-таймаут, retry по матрице (503 и connect — да, 4xx и read-timeout — нет), backoff с джиттером.

**Дальше**: урок 5.4 — llama-cpp-python: тот же движок без демона, прямо в вашем процессе.

## Полезные ссылки

- Ollama API: <https://github.com/ollama/ollama/blob/main/docs/api.md>
- FAQ Ollama (все переменные окружения): <https://github.com/ollama/ollama/blob/main/docs/faq.md>
- NSSM: <https://nssm.cc/usage>
- systemd unit-файлы: <https://www.freedesktop.org/software/systemd/man/latest/systemd.service.html>
- tenacity: <https://tenacity.readthedocs.io>

---

## Решения упражнений

### Упражнение 1: прогрев после рестарта

In [ ]:
import time


def warmup(models: list[str], keep_alive: str = "1h") -> None:
    """Загружает модели в RAM после рестарта демона (гасит холодный старт)."""
    for model in models:
        t0 = time.perf_counter()
        try:
            resp = httpx.post(
                f"{OLLAMA_URL}/api/generate",
                json={"model": model, "prompt": "ping", "stream": False,
                      "keep_alive": keep_alive, "options": {"num_predict": 1}},
                timeout=300,
            )
            resp.raise_for_status()
            load_s = resp.json().get("load_duration", 0) / NS
            print(f"{model:<20} прогрета за {time.perf_counter() - t0:.1f} с "
                  f"(загрузка с диска: {load_s:.1f} с)")
        except httpx.HTTPError as exc:
            print(f"{model:<20} ошибка прогрева: {exc}")


if OLLAMA_UP:
    warmup([MODEL])
else:
    print("Пропущено: Ollama не запущена")

### Упражнение 2: retry с джиттером руками

In [ ]:
import functools
import random
import time


def retry_with_jitter(attempts: int = 10, base_delay: float = 0.5):
    """Повторяет вызов при ConnectError/503 с экспоненциальным backoff и джиттером."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_exc: BaseException | None = None
            for attempt in range(attempts):
                try:
                    return func(*args, **kwargs)
                except (httpx.ConnectError, httpx.ConnectTimeout, OllamaHTTPError) as exc:
                    retryable = not isinstance(exc, OllamaHTTPError) or exc.status_code in (502, 503)
                    if not retryable:
                        raise
                    last_exc = exc
                    if attempt < attempts - 1:
                        delay = base_delay * (2 ** attempt) * random.uniform(0.5, 1.5)
                        print(f"попытка {attempt + 1} не удалась ({exc.__class__.__name__}), "
                              f"жду {delay:.2f} с")
                        time.sleep(delay)
            raise last_exc  # все попытки исчерпаны
        return wrapper
    return decorator


@retry_with_jitter(attempts=10)
def flaky_call() -> str:
    # Имитация нестабильного апстрима: 2 из 3 вызовов «падают» 503-й
    if random.random() < 0.66:
        raise OllamaHTTPError(503, "server busy")
    return "успех"


random.seed(10)  # подобран так, чтобы увидеть оба повтора
print(flaky_call())

### Упражнение 3: конфиг под сценарий

```
OLLAMA_KEEP_ALIVE=-1        # поток постоянный, выделенный сервер: пусть модель живёт в RAM всегда
OLLAMA_NUM_PARALLEL=1       # CPU: throughput не вырастет, а KV-кеш и латентность каждого — да
OLLAMA_MAX_LOADED_MODELS=1  # модель одна; вторая случайная (кто-то дернул другую) не должна вытеснять боевую
OLLAMA_MAX_QUEUE=64         # клиенты умеют retry: лучше быстрый 503, чем минуты в очереди;
                            # 64 — при 8 ток/с и ответах ~200 токенов это уже ~25 минут работы
OLLAMA_CONTEXT_LENGTH=4096  # фиксируем один контекст для всех, чтобы runner не перезапускался;
                            # 4k при 7B-Q4 держит RAM в ~6 ГБ из 16 с запасом под ОС и пики
```

Отдельно стоит договориться с клиентами не передавать свой `num_ctx` в options — иначе фиксация контекста не сработает.